# 2. Pré-processamento e Limpeza de Dados

**Objetivo:** Tratar os dados brutos de Smart TVs (`data/dados_brutos.csv`), convertendo preços para o formato numérico (`float`), removendo ruídos/duplicatas e extraindo a marca de cada televisor para viabilizar a análise estatística.

**Metas desta etapa:**
1. Carregar `data/dados_brutos.csv`
2. Identificar e tratar valores ausentes ou duplicados
3. Converter a coluna `preco_bruto` (ex: `R$ 3.279,00`) para a coluna `preco` (`float`)
4. Extrair e padronizar a coluna `marca` (Samsung, LG, TCL, AOC, Philips, etc.)
5. Salvar o resultado final em `data/dados_tratados.csv`

In [1]:
import pandas as pd
import numpy as np
import re
import os

# Carregar dados brutos
caminho_bruto = "../data/dados_brutos.csv"
df = pd.read_csv(caminho_bruto)

print(f"✅ Base bruta carregada com sucesso!")
print(f"• Total de registros: {len(df)}")
print(f"• Colunas encontradas: {list(df.columns)}\n")

# Exibir os 5 primeiros registros e tipos de dados
df.head()

✅ Base bruta carregada com sucesso!
• Total de registros: 44
• Colunas encontradas: ['id_coleta', 'produto', 'preco_bruto', 'loja_oferta', 'url', 'categoria']



,id_coleta,produto,preco_bruto,loja_oferta,url,categoria
0,1,"Smart TV Mini LED 55"" TCL 4K 55C6K","R$ 3.279,00",Via Amazon,NaN,Smart TV
1,2,"Smart TV Mini LED 55"" TCL 4K 55C6K","R$ 3.279,00",Via Amazon,https://www.buscape.com.br/tv/smart-tv-mini-le...,Smart TV
2,3,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K","R$ 3.963,60",Via Webcontinental,NaN,Smart TV
3,4,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K","R$ 3.963,60",Via Webcontinental,https://www.buscape.com.br/tv/smart-tv-qd-mini...,Smart TV
4,5,"Smart TV LED 50"" LG 4K UA7500","R$ 2.230,41",Via Magazine Luiza,NaN,Smart TV


## 2.1 Tratamento e Conversão da Coluna de Preço

Transformação da coluna `preco_bruto` (texto) para `preco` (numérico/float):
* Remoção dos caracteres `R$`, espaços e pontos de milhar
* Substituição da vírgula decimal por ponto (`,` $\rightarrow$ `.`)
* Conversão de tipo de dado para `float64`

In [2]:
def limpar_preco(preco_str):
    if pd.isna(preco_str) or preco_str == "N/A":
        return np.nan
    
    # 1. Manter apenas números, pontos e vírgulas
    texto_limpo = re.sub(r'[^\d,\.]', '', str(preco_str))
    
    # 2. Remover pontos de milhar e trocar vírgula por ponto decimal
    texto_limpo = texto_limpo.replace('.', '').replace(',', '.')
    
    try:
        return float(texto_limpo)
    except ValueError:
        return np.nan

# Aplicar a limpeza criando a coluna 'preco'
df['preco'] = df['preco_bruto'].apply(limpar_preco)

print("✅ Conversão de preço realizada com sucesso!")
print(f"• Preços convertidos para numérico (float64)")
print(f"• Quantidade de preços nulos (NaN): {df['preco'].isna().sum()}\n")

# Exibir resumo estatístico rápido dos preços para validar a conversão
print("--- Resumo dos Preços (R$) ---")
print(df['preco'].describe().round(2))

# Visualizar comparação antes vs depois
df[['produto', 'preco_bruto', 'preco']].head()

✅ Conversão de preço realizada com sucesso!
• Preços convertidos para numérico (float64)
• Quantidade de preços nulos (NaN): 0

--- Resumo dos Preços (R$) ---
count       44.00
mean      3957.97
std       2746.50
min        889.00
25%       2075.95
50%       2826.96
75%       5859.73
max      12706.80
Name: preco, dtype: float64


,produto,preco_bruto,preco
0,"Smart TV Mini LED 55"" TCL 4K 55C6K","R$ 3.279,00",3279.00
1,"Smart TV Mini LED 55"" TCL 4K 55C6K","R$ 3.279,00",3279.00
2,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K","R$ 3.963,60",3963.60
3,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K","R$ 3.963,60",3963.60
4,"Smart TV LED 50"" LG 4K UA7500","R$ 2.230,41",2230.41


## 2.2 Extração e Padronização da Marca

Para responder à pergunta de negócio, precisamos isolar a fabricante do televisor a partir da coluna `produto`. 
Identificaremos as marcas principais do mercado brasileiro de Smart TVs: **Samsung, LG, TCL, AOC, Philips, Philco, Toshiba, Sony, Panasonic, Aiwa e Multilaser**.

In [3]:
# Lista de marcas conhecidas de televisores no mercado brasileiro
MARCAS_CONHECIDAS = [
    "Samsung", "LG", "TCL", "AOC", "Philips", 
    "Philco", "Toshiba", "Sony", "Panasonic", 
    "Aiwa", "Multilaser", "Semp"
]

def extrair_marca(nome_produto):
    if pd.isna(nome_produto) or nome_produto == "N/A":
        return "Outras"
    
    nome_lower = str(nome_produto).lower()
    
    # Busca a marca dentro do texto do produto
    for marca in MARCAS_CONHECIDAS:
        # Busca por palavra inteira para evitar falso-positivo
        if re.search(r'\b' + re.escape(marca.lower()) + r'\b', nome_lower):
            return marca
            
    return "Outras"

# Aplicar a extração
df['marca'] = df['produto'].apply(extrair_marca)

print("✅ Extração de marcas concluída com sucesso!\n")
print("--- Distribuição de Produtos por Marca ---")
print(df['marca'].value_counts())

# Exibir amostra dos dados com a nova coluna
df[['produto', 'marca', 'preco']].head(10)

✅ Extração de marcas concluída com sucesso!

--- Distribuição de Produtos por Marca ---
marca
Outras     14
TCL        12
Samsung    10
LG          6
Philco      2
Name: count, dtype: int64


,produto,marca,preco
0,"Smart TV Mini LED 55"" TCL 4K 55C6K",TCL,3279.00
1,"Smart TV Mini LED 55"" TCL 4K 55C6K",TCL,3279.00
2,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K",TCL,3963.60
3,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K",TCL,3963.60
4,"Smart TV LED 50"" LG 4K UA7500",LG,2230.41
5,"Smart TV LED 50"" LG 4K UA7500",LG,2230.41
6,"Smart TV QLED 50"" TCL 4K P7K",TCL,2069.10
7,"Smart TV QLED 50"" TCL 4K P7K",TCL,2069.10
8,"Smart TV QD-Mini LED 75"" TCL 4K C6K",TCL,5859.73
9,"Smart TV QD-Mini LED 75"" TCL 4K C6K",TCL,5859.73


## 2.3 Tratamento de Inconsistências e Exportação

Validação final da base de dados:
* Remoção de linhas sem preço válido (valores `NaN`)
* Eliminação de produtos duplicados
* Seleção e reordenação dos campos tratados
* Salvar os dados limpos em `data/dados_tratados.csv`

In [4]:
# Backup da quantidade inicial
total_inicial = len(df)

# 1. Remover registros sem preço válido
df_limpo = df.dropna(subset=['preco']).copy()

# 2. Remover produtos duplicados pelo nome do produto
df_limpo = df_limpo.drop_duplicates(subset=['produto']).copy()

# 3. Selecionar e organizar colunas finais
colunas_finais = ['id_coleta', 'marca', 'produto', 'preco', 'loja_oferta', 'url', 'categoria']
df_tratado = df_limpo[colunas_finais].reset_index(drop=True)

# Recalcular ID limpo
df_tratado['id_produto'] = df_tratado.index + 1
df_tratado = df_tratado[['id_produto', 'marca', 'produto', 'preco', 'loja_oferta', 'url', 'categoria']]

# 4. Salvar o arquivo final em data/dados_tratados.csv
caminho_tratado = "../data/dados_tratados.csv"
df_tratado.to_csv(caminho_tratado, index=False, encoding="utf-8-sig")

print("✅ Módulo 2 de Limpeza e Tratamento concluído com sucesso!\n")
print(f"• Registros originais: {total_inicial}")
print(f"• Registros finais válidos: {len(df_tratado)}")
print(f"• Arquivo salvo em: {caminho_tratado}\n")

# Resumo final por Marca
print("--- Preço Médio e Quantidade por Marca (Base Tratada) ---")
resumo_marcas = df_tratado.groupby('marca')['preco'].agg(['count', 'mean', 'min', 'max']).round(2)
resumo_marcas = resumo_marcas.rename(columns={'count': 'qtd_tv', 'mean': 'preco_medio', 'min': 'preco_min', 'max': 'preco_max'})
print(resumo_marcas)

# Exibir os 5 primeiros registros tratados
df_tratado.head()

✅ Módulo 2 de Limpeza e Tratamento concluído com sucesso!

• Registros originais: 44
• Registros finais válidos: 17
• Arquivo salvo em: ../data/dados_tratados.csv

--- Preço Médio e Quantidade por Marca (Base Tratada) ---
         qtd_tv  preco_medio  preco_min  preco_max
marca                                             
LG            3      4778.04    2230.41    6252.66
Outras        2      2250.03    2198.00    2302.06
Philco        1       889.00     889.00     889.00
Samsung       5      3585.08    1612.80    6031.55
TCL           6      4120.70    2069.10    5984.05


,id_produto,marca,produto,preco,loja_oferta,url,categoria
0,1,TCL,"Smart TV Mini LED 55"" TCL 4K 55C6K",3279.00,Via Amazon,NaN,Smart TV
1,2,TCL,"Smart TV QD-Mini LED 65"" TCL 4K 65C6K",3963.60,Via Webcontinental,NaN,Smart TV
2,3,LG,"Smart TV LED 50"" LG 4K UA7500",2230.41,Via Magazine Luiza,NaN,Smart TV
3,4,TCL,"Smart TV QLED 50"" TCL 4K P7K",2069.10,Via Magazine Luiza,NaN,Smart TV
4,5,TCL,"Smart TV QD-Mini LED 75"" TCL 4K C6K",5859.73,Via Magazine Luiza,NaN,Smart TV
